In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [55]:
## Parameter 
start_date = '20231002'
end_date = '20231008'

In [56]:
## first ride customers 

weekly_view_data = f"""
    
    WITH base_gross_order AS (

    SELECT  
        yyyymmdd,
        customer_id,
        customer_obj_mobile customer_mobile,
        order_id,
        order_requested_epoch requested,
        accepted_epoch onTheWay,
        arrived_epoch arrived,
        started_epoch started,
        dropped_epoch dropped,
        CASE WHEN dropped_epoch IS NULL THEN COALESCE(expired_epoch, customer_cancelled_epoch)
        WHEN (CAST(dropped_epoch AS DOUBLE) /1000) < (CAST(updated_epoch AS DOUBLE) /1000) AND ((CAST(dropped_epoch AS DOUBLE) + 900000) /1000) >= (CAST(updated_epoch AS DOUBLE) /1000) THEN updated_epoch
        ELSE dropped_epoch END AS end_state,
        COALESCE(expired_epoch, customer_cancelled_epoch) incompleted_state,
        modified_order_status order_status,
        CAST(hour AS INT) session_start,
        (CAST(hour AS INT) + 1) session_end
        
    FROM 
        orders.order_logs_fact 
    
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        -- AND city_name = 'Chennai'
    ),
    
    postorderstatus AS (
        
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        epoch postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_requesting_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        updated_epoch postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_ontheway_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        updated_epoch postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_arrived_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        epoch postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_started_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        epoch postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_dropped_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    ),
    
    screendropoff AS (
    
    SELECT  
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        CAST(CAST(event_props_ct_session_id AS DECIMAL) AS VARCHAR) ct_session_id,
        updated_epoch screendropoff
    FROM 
        clevertap.customer_screendropoff_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND event_props_screen = 'PostOrderActivity'
    ),
    
    merge_datasets AS (
    
    SELECT 
        yyyymmdd,
        customer_id,
        order_id,
        order_status,
        requested,
        onTheWay,
        arrived,
        started,
        dropped,
        end_state,
        
        AVG(postorderstatus_requesting) postorderstatus_requesting,
        AVG(postorderstatus_ontheway) postorderstatus_ontheway,
        AVG(postorderstatus_arrived) postorderstatus_arrived,
        AVG(postorderstatus_started) postorderstatus_started,
        AVG(postorderstatus_dropped) postorderstatus_dropped,
        
        MAX(screendropoff_before_ontheway) screendropoff_before_ontheway,
        MAX(screendropoff_before_arrived) screendropoff_before_arrived,
        MAX(screendropoff_before_started) screendropoff_before_started,
        MAX(screendropoff_before_dropped) screendropoff_before_dropped,
        MAX(screendropoff_after_dropped) screendropoff_after_dropped,
        
        COUNT(DISTINCT screendropoff_before_ontheway) count_screendropoff_before_ontheway,
        COUNT(DISTINCT screendropoff_before_arrived) count_screendropoff_before_arrived,
        COUNT(DISTINCT screendropoff_before_started) count_screendropoff_before_started,
        COUNT(DISTINCT screendropoff_before_dropped) count_screendropoff_before_dropped,
        COUNT(DISTINCT screendropoff_after_dropped) count_screendropoff_after_dropped
        
    FROM 
        (
        SELECT 
            base.yyyymmdd,
            base.customer_id,
            base.customer_mobile,
            base.order_id,
            
            base.requested,
            COALESCE(pos.postorderstatus_requesting,base.requested) postorderstatus_requesting,
            CASE WHEN sdo.screendropoff >= base.requested AND (sdo.screendropoff < base.onTheWay OR sdo.screendropoff < pos.postorderstatus_ontheway) THEN sdo.screendropoff END screendropoff_before_ontheway,
            CASE WHEN base.incompleted_state >= base.requested AND base.incompleted_state < COALESCE(base.onTheWay, 0) THEN base.incompleted_state END end_before_onTheWay,
            
            base.onTheWay,
            pos.postorderstatus_ontheway,
            CASE WHEN (sdo.screendropoff >= base.onTheWay OR sdo.screendropoff >= pos.postorderstatus_ontheway) AND sdo.screendropoff < base.arrived THEN sdo.screendropoff END screendropoff_before_arrived,
            CASE WHEN base.incompleted_state >= base.onTheWay AND base.incompleted_state < COALESCE(base.arrived, 0) THEN base.incompleted_state END end_before_arrivded, 
            
            base.arrived,
            pos.postorderstatus_arrived,
            CASE WHEN (sdo.screendropoff >= base.arrived OR sdo.screendropoff >= pos.postorderstatus_arrived) AND sdo.screendropoff < base.started THEN sdo.screendropoff END screendropoff_before_started,
            CASE WHEN base.incompleted_state >= base.arrived AND base.incompleted_state < COALESCE(base.started, 0) THEN base.incompleted_state END end_after_arrivded, 
            
            base.started,
            pos.postorderstatus_started,
            CASE WHEN sdo.screendropoff >= base.started AND sdo.screendropoff < COALESCE(base.dropped, 0) THEN sdo.screendropoff END screendropoff_before_dropped,
            
            base.dropped,
            pos.postorderstatus_dropped,
            CASE WHEN sdo.screendropoff >= base.dropped AND sdo.screendropoff < base.end_state THEN sdo.screendropoff END screendropoff_after_dropped,
            
            base.end_state,
            base.order_status

        FROM 
            base_gross_order base
        LEFT JOIN 
            postorderstatus pos 
            ON base.yyyymmdd = pos.yyyymmdd
            AND base.order_id = pos.order_id
        LEFT JOIN 
            screendropoff sdo
            ON pos.yyyymmdd = sdo.yyyymmdd
            AND pos.customer_id = sdo.customer_id
            AND pos.customer_mobile = sdo.customer_mobile
            -- AND pos.ct_session_id = sdo.ct_session_id
        
        -- WHERE 
            -- base.order_id = '655b5956cbbabb6828039bbd'
        )
    GROUP BY 1,2,3,4,5,6,7,8,9,10
    ),
    
    calc_screen_time AS (
    
    SELECT
        *,
        CASE 
        WHEN requested_time = 0 THEN 0 
        WHEN time_requested_screen > requested_time THEN 100.0
        ELSE COALESCE(TRY(time_requested_screen*100.0/requested_time), 0) END AS requested_screen_percentage,

        CASE 
        WHEN onTheWay_time = 0 THEN 0 
        WHEN time_onTheWay_screen > onTheWay_time THEN 100.0
        ELSE COALESCE(TRY(time_onTheWay_screen*100.0/onTheWay_time), 0) END AS onTheWay_screen_percentage,
        
        CASE 
        WHEN arrived_time = 0 THEN 0 
        WHEN time_arrived_screen > arrived_time THEN 100.0
        ELSE COALESCE(TRY(time_arrived_screen*100.0/arrived_time), 0) END AS arrived_screen_percentage,
        
        CASE 
        WHEN started_time = 0 THEN 0 
        WHEN time_started_screen > started_time THEN 100.0
        ELSE COALESCE(TRY(time_started_screen*100.0/started_time), 0) END AS started_screen_percentage,
        
        CASE 
        WHEN ride_timing = 0 THEN 0
        WHEN (time_requested_screen + time_onTheWay_screen + time_arrived_screen + time_started_screen + time_dropped_screen) > ride_timing THEN 0
        ELSE COALESCE(TRY((time_requested_screen + time_onTheWay_screen + time_arrived_screen + time_started_screen + time_dropped_screen)*100.0/ride_timing),0) END AS screen_percentage
        
    FROM
        (
        SELECT
            merge_datasets.*,
            
            COALESCE(CASE 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NOT NULL AND screendropoff_before_ontheway <= onTheWay THEN abs(screendropoff_before_ontheway/1000 - requested/1000) 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL AND onTheWay IS NOT NULL AND postorderstatus_requesting <= onTheWay THEN abs(onTheWay/1000 - requested/1000) 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL AND onTheWay IS NULL THEN abs(end_state/1000 - requested/1000) 
                END, 0) time_requested_screen,
                
            COALESCE((COALESCE(onTheWay,end_state)/1000 - requested/1000), 0) requested_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NOT NULL AND screendropoff_before_arrived <= arrived THEN abs(screendropoff_before_arrived/1000 - postorderstatus_ontheway/1000) 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL AND arrived IS NOT NULL AND postorderstatus_ontheway <= arrived THEN abs(arrived/1000 - postorderstatus_ontheway/1000) 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL AND arrived IS NULL THEN abs(end_state/1000 - postorderstatus_ontheway/1000) 
                END, 0) time_onTheWay_screen,
                
            COALESCE((COALESCE(arrived,end_state)/1000 - onTheWay/1000), 0) onTheWay_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NOT NULL AND screendropoff_before_started <= started THEN abs(screendropoff_before_started/1000 - postorderstatus_arrived/1000) 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL AND started IS NOT NULL AND postorderstatus_arrived <= started THEN abs(started/1000 - postorderstatus_arrived/1000) 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL AND started IS NULL THEN abs(end_state/1000 - postorderstatus_arrived/1000) 
                END, 0) time_arrived_screen,
                
            COALESCE((COALESCE(started,end_state)/1000 - arrived/1000), 0) arrived_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NOT NULL AND screendropoff_before_dropped <= dropped THEN abs(screendropoff_before_dropped/1000 - postorderstatus_started/1000) 
                WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NULL AND dropped IS NOT NULL AND postorderstatus_started <= dropped THEN abs(dropped/1000 - postorderstatus_started/1000)
                END, 0) time_started_screen,
                
            COALESCE((COALESCE(dropped,end_state)/1000 - started/1000), 0) started_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_dropped IS NOT NULL AND screendropoff_after_dropped IS NOT NULL AND screendropoff_after_dropped > dropped THEN abs(screendropoff_after_dropped/1000 - postorderstatus_dropped/1000) 
                WHEN postorderstatus_dropped IS NOT NULL AND screendropoff_after_dropped IS NULL AND dropped IS NOT NULL AND postorderstatus_dropped > dropped THEN 15 
                END, 0) time_dropped_screen,
                
            ((GREATEST(COALESCE(screendropoff_after_dropped,0),dropped,end_state)/1000) - (requested/1000)) AS ride_timing,
            COALESCE((count_screendropoff_before_ontheway + count_screendropoff_before_arrived + count_screendropoff_before_started + count_screendropoff_before_dropped + count_screendropoff_after_dropped), 0) screendropoff
    
        FROM 
            merge_datasets
        )
    )
    
    SELECT
        
        WEEK(DATE_PARSE(yyyymmdd,'%Y%m%d')) as week_number,
        
        MIN(yyyymmdd) || ' to ' || MAX(yyyymmdd) date_range,
        
        -- requested
        COUNT(DISTINCT order_id) requested,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_requesting IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age viewed on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age no dropOff on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_ontheway IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age 2-3 dropOff on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_ontheway > 3 THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age 3+ dropOff on requested screen",
        
        (AVG(time_requested_screen))  "avg_time_on_requested_screen (sec)",
        (AVG(requested_screen_percentage)) "requested_screen_engagement %",
        
        -- onTheWay
        COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END) onTheWay,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_ontheway IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age viewed on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age no dropOff on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_arrived IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_arrived > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on onTheWay screen",
        
        -- AVG(count_screendropoff_before_arrived) count_screendropoff_before_arrived,
        (AVG(time_onTheWay_screen))  "avg_time_on_onTheWay_screen (sec)",
        (AVG(onTheWay_screen_percentage)) "onTheWay_screen_engagement %",
        
        -- arrived
        COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END) arrived,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_arrived IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age viewed on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age no dropOff on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_started IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_started > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on arrived screen",
        
        (AVG(time_arrived_screen))  "avg_time_on_arrived_screen (sec)",
        (AVG(arrived_screen_percentage)) "arrived_screen_engagement %",
        
        -- started
        COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END) started,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_started IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age viewed on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age no dropOff on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_dropped IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_dropped > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on started screen",
        
        (AVG(time_started_screen))  "avg_time_on_started_screen (sec)",
        (AVG(started_screen_percentage)) "started_screen_engagement %",
        
        -- dropped
        COUNT(DISTINCT CASE WHEN dropped IS NOT NULL THEN order_id END) dropped,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_dropped IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN dropped IS NOT NULL THEN order_id END)), 0) "%age viewed on dropped screen",
        (AVG(time_dropped_screen))  "avg_time_on_dropped_screen (sec)",
        
        --other
        AVG(screen_percentage) "overall_screen_engagement %",
        AVG(screendropoff) "avg screendropoff #"
        
    FROM 
        calc_screen_time
    
    GROUP BY 1
    ORDER BY 1 DESC


"""


df_weekly_view_data = pd.read_sql(weekly_view_data, connection)
df_weekly_view_data

,week_number,date_range,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,40,20231002 to 20231008,15402335,100.0,90.4,3.5,2.4,88.174099,91.1,11736822,74.1,27.3,20.0,12.9,136.790375,39.743089,9234513,50.9,29.0,9.0,1.7,24.585238,17.553952,8634411,49.7,6.4,18.6,7.5,100.220281,9.034585,8622114,33.8,4.37345,14.981268,2.203301


In [58]:
df1_weekly_oct2tooct8

,week_number,date_range,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,40,20231002 to 20231008,15402335,100.0,90.4,3.5,2.4,88.17,91.1,11736822,74.1,27.3,20.0,12.9,136.79,39.74,9234513,50.9,29.0,9.0,1.7,24.59,17.55,8634411,49.7,6.4,18.6,7.5,100.22,9.03,8622114,33.8,4.37,14.98,2.2


In [54]:
df1_weekly_oct9tooct15

,week_number,date_range,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,41,20231009 to 20231015,16782486,100.0,90.8,3.3,2.3,97.04,91.3,12185881,74.2,27.7,19.9,13.0,131.82,37.91,9538040,51.0,28.9,9.1,1.8,23.58,16.62,8914870,49.3,6.5,18.4,7.4,94.3,8.47,8902870,33.4,4.11,14.23,2.09


In [50]:
df1_weekly_oct16tooct22

,week_number,date_range,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,42,20231016 to 20231022,17089584,100.0,90.5,3.5,2.4,96.63,90.9,12486516,74.7,28.3,19.8,13.2,135.57,38.46,9683766,51.5,29.4,9.1,1.8,23.57,16.71,9046711,50.2,6.6,18.7,7.7,98.7,8.63,9034143,33.8,4.18,14.32,2.14


In [46]:
df1_weekly_oct23tooct29

,week_number,date_range,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,43,20231023 to 20231029,15358410,100.0,90.9,3.3,2.3,95.36,91.1,11050634,73.5,27.8,19.6,12.6,128.78,37.33,8670041,50.8,29.2,8.9,1.7,23.04,16.51,8101092,49.3,6.5,18.2,7.3,92.6,8.44,8090500,33.4,4.17,14.04,2.04


In [59]:
frames = [df1_weekly_oct2tooct8, df1_weekly_oct9tooct15, df1_weekly_oct16tooct22, df1_weekly_oct23tooct29]
result = pd.concat(frames)

In [60]:
result

,week_number,date_range,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,40,20231002 to 20231008,15402335,100.0,90.4,3.5,2.4,88.17,91.1,11736822,74.1,27.3,20.0,12.9,136.79,39.74,9234513,50.9,29.0,9.0,1.7,24.59,17.55,8634411,49.7,6.4,18.6,7.5,100.22,9.03,8622114,33.8,4.37,14.98,2.20
0,41,20231009 to 20231015,16782486,100.0,90.8,3.3,2.3,97.04,91.3,12185881,74.2,27.7,19.9,13.0,131.82,37.91,9538040,51.0,28.9,9.1,1.8,23.58,16.62,8914870,49.3,6.5,18.4,7.4,94.30,8.47,8902870,33.4,4.11,14.23,2.09
0,42,20231016 to 20231022,17089584,100.0,90.5,3.5,2.4,96.63,90.9,12486516,74.7,28.3,19.8,13.2,135.57,38.46,9683766,51.5,29.4,9.1,1.8,23.57,16.71,9046711,50.2,6.6,18.7,7.7,98.70,8.63,9034143,33.8,4.18,14.32,2.14
0,43,20231023 to 20231029,15358410,100.0,90.9,3.3,2.3,95.36,91.1,11050634,73.5,27.8,19.6,12.6,128.78,37.33,8670041,50.8,29.2,8.9,1.7,23.04,16.51,8101092,49.3,6.5,18.2,7.3,92.60,8.44,8090500,33.4,4.17,14.04,2.04


In [61]:
result.to_clipboard(index=False)

In [57]:
df1_weekly_oct2tooct8 = df_weekly_view_data.round(2)

In [62]:
## Parameter 
start_date = '20231002'
end_date = '20231008'

In [65]:
## first ride customers 

segment_view_data = f"""

WITH base_gross_order AS (

    SELECT  
        yyyymmdd,
        customer_id,
        customer_obj_mobile customer_mobile,
        order_id,
        order_requested_epoch requested,
        accepted_epoch onTheWay,
        arrived_epoch arrived,
        started_epoch started,
        dropped_epoch dropped,
        CASE WHEN dropped_epoch IS NULL THEN COALESCE(expired_epoch, customer_cancelled_epoch)
        WHEN (CAST(dropped_epoch AS DOUBLE) /1000) < (CAST(updated_epoch AS DOUBLE) /1000) AND ((CAST(dropped_epoch AS DOUBLE) + 900000) /1000) >= (CAST(updated_epoch AS DOUBLE) /1000) THEN updated_epoch
        ELSE dropped_epoch END AS end_state,
        COALESCE(expired_epoch, customer_cancelled_epoch) incompleted_state,
        modified_order_status order_status,
        CAST(hour AS INT) session_start,
        (CAST(hour AS INT) + 1) session_end,
        CASE WHEN service_obj_service_name IN ('Link', 'Auto', 'CabEconomy','Auto Pool') THEN service_obj_service_name ELSE 'Other service' END AS service_name
        
    FROM 
        orders.order_logs_fact 
    
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        -- AND city_name = 'Chennai'
    ),
    
    postorderstatus AS (
        
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        epoch postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_requesting_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        updated_epoch postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_ontheway_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        updated_epoch postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_arrived_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        epoch postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_started_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        epoch postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_dropped_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    ),
    
    screendropoff AS (
    
    SELECT  
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        CAST(CAST(event_props_ct_session_id AS DECIMAL) AS VARCHAR) ct_session_id,
        updated_epoch screendropoff
    FROM 
        clevertap.customer_screendropoff_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND event_props_screen = 'PostOrderActivity'
    ),
    
    merge_datasets AS (
    
    SELECT 
        yyyymmdd,
        service_name,
        customer_id,
        order_id,
        order_status,
        requested,
        onTheWay,
        arrived,
        started,
        dropped,
        end_state,
        
        AVG(postorderstatus_requesting) postorderstatus_requesting,
        AVG(postorderstatus_ontheway) postorderstatus_ontheway,
        AVG(postorderstatus_arrived) postorderstatus_arrived,
        AVG(postorderstatus_started) postorderstatus_started,
        AVG(postorderstatus_dropped) postorderstatus_dropped,
        
        MAX(screendropoff_before_ontheway) screendropoff_before_ontheway,
        MAX(screendropoff_before_arrived) screendropoff_before_arrived,
        MAX(screendropoff_before_started) screendropoff_before_started,
        MAX(screendropoff_before_dropped) screendropoff_before_dropped,
        MAX(screendropoff_after_dropped) screendropoff_after_dropped,
        
        COUNT(DISTINCT screendropoff_before_ontheway) count_screendropoff_before_ontheway,
        COUNT(DISTINCT screendropoff_before_arrived) count_screendropoff_before_arrived,
        COUNT(DISTINCT screendropoff_before_started) count_screendropoff_before_started,
        COUNT(DISTINCT screendropoff_before_dropped) count_screendropoff_before_dropped,
        COUNT(DISTINCT screendropoff_after_dropped) count_screendropoff_after_dropped
        
    FROM 
        (
        SELECT 
            base.yyyymmdd,
            base.service_name,
            base.customer_id,
            base.customer_mobile,
            base.order_id,
            
            base.requested,
            COALESCE(pos.postorderstatus_requesting,base.requested) postorderstatus_requesting,
            CASE WHEN sdo.screendropoff >= base.requested AND (sdo.screendropoff < base.onTheWay OR sdo.screendropoff < pos.postorderstatus_ontheway) THEN sdo.screendropoff END screendropoff_before_ontheway,
            CASE WHEN base.incompleted_state >= base.requested AND base.incompleted_state < COALESCE(base.onTheWay, 0) THEN base.incompleted_state END end_before_onTheWay,
            
            base.onTheWay,
            pos.postorderstatus_ontheway,
            CASE WHEN (sdo.screendropoff >= base.onTheWay OR sdo.screendropoff >= pos.postorderstatus_ontheway) AND sdo.screendropoff < base.arrived THEN sdo.screendropoff END screendropoff_before_arrived,
            CASE WHEN base.incompleted_state >= base.onTheWay AND base.incompleted_state < COALESCE(base.arrived, 0) THEN base.incompleted_state END end_before_arrivded, 
            
            base.arrived,
            pos.postorderstatus_arrived,
            CASE WHEN (sdo.screendropoff >= base.arrived OR sdo.screendropoff >= pos.postorderstatus_arrived) AND sdo.screendropoff < base.started THEN sdo.screendropoff END screendropoff_before_started,
            CASE WHEN base.incompleted_state >= base.arrived AND base.incompleted_state < COALESCE(base.started, 0) THEN base.incompleted_state END end_after_arrivded, 
            
            base.started,
            pos.postorderstatus_started,
            CASE WHEN sdo.screendropoff >= base.started AND sdo.screendropoff < COALESCE(base.dropped, 0) THEN sdo.screendropoff END screendropoff_before_dropped,
            
            base.dropped,
            pos.postorderstatus_dropped,
            CASE WHEN sdo.screendropoff >= base.dropped AND sdo.screendropoff < base.end_state THEN sdo.screendropoff END screendropoff_after_dropped,
            
            base.end_state,
            base.order_status

        FROM 
            base_gross_order base
        LEFT JOIN 
            postorderstatus pos 
            ON base.yyyymmdd = pos.yyyymmdd
            AND base.order_id = pos.order_id
        LEFT JOIN 
            screendropoff sdo
            ON pos.yyyymmdd = sdo.yyyymmdd
            AND pos.customer_id = sdo.customer_id
            AND pos.customer_mobile = sdo.customer_mobile
            -- AND pos.ct_session_id = sdo.ct_session_id
        
        -- WHERE 
            -- base.order_id = '655b5956cbbabb6828039bbd'
        )
    GROUP BY 1,2,3,4,5,6,7,8,9,10,11
    ),
    
    calc_screen_time AS (
    
    SELECT
        *,
        CASE 
        WHEN requested_time = 0 THEN 0 
        WHEN time_requested_screen > requested_time THEN 100.0
        ELSE COALESCE(TRY(time_requested_screen*100.0/requested_time), 0) END AS requested_screen_percentage,

        CASE 
        WHEN onTheWay_time = 0 THEN 0 
        WHEN time_onTheWay_screen > onTheWay_time THEN 100.0
        ELSE COALESCE(TRY(time_onTheWay_screen*100.0/onTheWay_time), 0) END AS onTheWay_screen_percentage,
        
        CASE 
        WHEN arrived_time = 0 THEN 0 
        WHEN time_arrived_screen > arrived_time THEN 100.0
        ELSE COALESCE(TRY(time_arrived_screen*100.0/arrived_time), 0) END AS arrived_screen_percentage,
        
        CASE 
        WHEN started_time = 0 THEN 0 
        WHEN time_started_screen > started_time THEN 100.0
        ELSE COALESCE(TRY(time_started_screen*100.0/started_time), 0) END AS started_screen_percentage,
        
        CASE 
        WHEN ride_timing = 0 THEN 0
        WHEN (time_requested_screen + time_onTheWay_screen + time_arrived_screen + time_started_screen + time_dropped_screen) > ride_timing THEN 0
        ELSE COALESCE(TRY((time_requested_screen + time_onTheWay_screen + time_arrived_screen + time_started_screen + time_dropped_screen)*100.0/ride_timing),0) END AS screen_percentage
        
    FROM
        (
        SELECT
            merge_datasets.*,
            
            COALESCE(CASE 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NOT NULL AND screendropoff_before_ontheway <= onTheWay THEN abs(screendropoff_before_ontheway/1000 - requested/1000) 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL AND onTheWay IS NOT NULL AND postorderstatus_requesting <= onTheWay THEN abs(onTheWay/1000 - requested/1000) 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL AND onTheWay IS NULL THEN abs(end_state/1000 - requested/1000) 
                END, 0) time_requested_screen,
                
            COALESCE((COALESCE(onTheWay,end_state)/1000 - requested/1000), 0) requested_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NOT NULL AND screendropoff_before_arrived <= arrived THEN abs(screendropoff_before_arrived/1000 - postorderstatus_ontheway/1000) 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL AND arrived IS NOT NULL AND postorderstatus_ontheway <= arrived THEN abs(arrived/1000 - postorderstatus_ontheway/1000) 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL AND arrived IS NULL THEN abs(end_state/1000 - postorderstatus_ontheway/1000) 
                END, 0) time_onTheWay_screen,
                
            COALESCE((COALESCE(arrived,end_state)/1000 - onTheWay/1000), 0) onTheWay_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NOT NULL AND screendropoff_before_started <= started THEN abs(screendropoff_before_started/1000 - postorderstatus_arrived/1000) 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL AND started IS NOT NULL AND postorderstatus_arrived <= started THEN abs(started/1000 - postorderstatus_arrived/1000) 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL AND started IS NULL THEN abs(end_state/1000 - postorderstatus_arrived/1000) 
                END, 0) time_arrived_screen,
                
            COALESCE((COALESCE(started,end_state)/1000 - arrived/1000), 0) arrived_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NOT NULL AND screendropoff_before_dropped <= dropped THEN abs(screendropoff_before_dropped/1000 - postorderstatus_started/1000) 
                WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NULL AND dropped IS NOT NULL AND postorderstatus_started <= dropped THEN abs(dropped/1000 - postorderstatus_started/1000)
                END, 0) time_started_screen,
                
            COALESCE((COALESCE(dropped,end_state)/1000 - started/1000), 0) started_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_dropped IS NOT NULL AND screendropoff_after_dropped IS NOT NULL AND screendropoff_after_dropped > dropped THEN abs(screendropoff_after_dropped/1000 - postorderstatus_dropped/1000) 
                WHEN postorderstatus_dropped IS NOT NULL AND screendropoff_after_dropped IS NULL AND dropped IS NOT NULL AND postorderstatus_dropped > dropped THEN 15 
                END, 0) time_dropped_screen,
                
            ((GREATEST(COALESCE(screendropoff_after_dropped,0),dropped,end_state)/1000) - (requested/1000)) AS ride_timing,
            COALESCE((count_screendropoff_before_ontheway + count_screendropoff_before_arrived + count_screendropoff_before_started + count_screendropoff_before_dropped + count_screendropoff_after_dropped), 0) screendropoff
    
        FROM 
            merge_datasets
        )
    )
    
    SELECT
        
        service_name,
        
        -- requested
        COUNT(DISTINCT order_id) requested,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_requesting IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age viewed on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age no dropOff on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_ontheway IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age 2-3 dropOff on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_ontheway > 3 THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age 3+ dropOff on requested screen",
        
        (AVG(time_requested_screen))  "avg_time_on_requested_screen (sec)",
        (AVG(requested_screen_percentage)) "requested_screen_engagement %",
        
        -- onTheWay
        COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END) onTheWay,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_ontheway IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age viewed on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age no dropOff on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_arrived IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_arrived > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on onTheWay screen",
        
        -- AVG(count_screendropoff_before_arrived) count_screendropoff_before_arrived,
        (AVG(time_onTheWay_screen))  "avg_time_on_onTheWay_screen (sec)",
        (AVG(onTheWay_screen_percentage)) "onTheWay_screen_engagement %",
        
        -- arrived
        COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END) arrived,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_arrived IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age viewed on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age no dropOff on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_started IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_started > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on arrived screen",
        
        (AVG(time_arrived_screen))  "avg_time_on_arrived_screen (sec)",
        (AVG(arrived_screen_percentage)) "arrived_screen_engagement %",
        
        -- started
        COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END) started,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_started IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age viewed on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age no dropOff on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_dropped IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_dropped > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on started screen",
        
        (AVG(time_started_screen))  "avg_time_on_started_screen (sec)",
        (AVG(started_screen_percentage)) "started_screen_engagement %",
        
        -- dropped
        COUNT(DISTINCT CASE WHEN dropped IS NOT NULL THEN order_id END) dropped,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_dropped IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN dropped IS NOT NULL THEN order_id END)), 0) "%age viewed on dropped screen",
        (AVG(time_dropped_screen))  "avg_time_on_dropped_screen (sec)",
        
        --other
        AVG(screen_percentage) "overall_screen_engagement %",
        AVG(screendropoff) "avg screendropoff #"
        
    FROM 
        calc_screen_time
    
    GROUP BY 1
    ORDER BY 2 DESC

"""


df_segment_view_data = pd.read_sql(segment_view_data, connection)
df_segment_view_data

,service_name,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,Link,7787203,100.0,88.8,4.2,2.8,74.671015,90.3,6606043,82.2,30.1,22.3,14.1,168.696351,48.198806,5206330,54.1,29.9,10.2,1.8,29.479172,20.564882,4856589,51.5,7.4,19.1,6.7,96.923150,10.134613,4850742,38.3,5.469731,18.141940,2.607126
1,Auto,5027142,100.0,89.2,3.9,2.7,108.738437,91.0,3708181,78.6,29.1,21.4,13.7,139.118792,42.092866,2795293,59.9,36.3,9.2,1.8,24.336828,19.457113,2597648,62.2,6.4,23.4,10.6,138.531691,10.710733,2595646,36.5,4.453726,15.298145,2.383132
2,Other service,2271713,100.0,98.5,0.5,0.4,89.896788,93.4,1243179,17.1,7.3,3.9,3.3,22.623167,6.631197,1100671,11.8,6.0,2.6,1.2,8.212576,3.537688,1056032,9.2,1.4,3.6,2.4,19.473993,1.709296,1051646,6.0,0.626316,3.970441,0.436924
3,CabEconomy,303557,100.0,91.9,3.0,2.4,78.450176,92.9,175917,77.0,29.0,19.3,18.1,137.866690,32.725362,130742,59.5,30.7,11.9,3.3,26.581819,14.246043,122775,60.2,7.4,22.0,13.4,157.010675,8.133350,122714,34.1,3.117983,11.518876,2.151484
4,Auto Pool,12720,100.0,97.9,0.7,0.6,151.831053,97.8,3502,73.3,46.7,10.9,9.9,47.545588,15.525997,1477,60.9,39.5,8.1,0.7,3.094720,4.258367,1367,68.8,6.8,26.9,22.2,42.971961,2.915251,1366,31.8,0.678774,3.872225,0.609355


In [66]:
## first ride customers 

ride_distance_view_data = f"""

WITH base_gross_order AS (

    SELECT  
        yyyymmdd,
        customer_id,
        customer_obj_mobile customer_mobile,
        order_id,
        order_requested_epoch requested,
        accepted_epoch onTheWay,
        arrived_epoch arrived,
        started_epoch started,
        dropped_epoch dropped,
        CASE WHEN dropped_epoch IS NULL THEN COALESCE(expired_epoch, customer_cancelled_epoch)
        WHEN (CAST(dropped_epoch AS DOUBLE) /1000) < (CAST(updated_epoch AS DOUBLE) /1000) AND ((CAST(dropped_epoch AS DOUBLE) + 900000) /1000) >= (CAST(updated_epoch AS DOUBLE) /1000) THEN updated_epoch
        ELSE dropped_epoch END AS end_state,
        COALESCE(expired_epoch, customer_cancelled_epoch) incompleted_state,
        modified_order_status order_status,
        CAST(hour AS INT) session_start,
        (CAST(hour AS INT) + 1) session_end,
        CASE
        WHEN ride_distance >= 5.0 THEN 'Long Distance' 
        WHEN ride_distance < 5.0 THEN 'Short Distance'
        ELSE 'Other' END AS ride_distance
    FROM 
        orders.order_logs_fact 
    
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        -- AND city_name = 'Chennai'
    ),
    
    postorderstatus AS (
        
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        epoch postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_requesting_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        updated_epoch postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_ontheway_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        updated_epoch postorderstatus_arrived,
        NULL postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_arrived_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        epoch postorderstatus_started,
        NULL postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_started_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        
    UNION
    
    SELECT 
        DISTINCT
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        json_extract_scalar(event_props,'$.orderId') order_id,
        NULL postorderstatus_requesting,
        NULL postorderstatus_ontheway,
        NULL postorderstatus_arrived,
        NULL postorderstatus_started,
        epoch postorderstatus_dropped
    FROM 
        clevertap.customer_postorderstatus_dropped_immutable
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
    ),
    
    screendropoff AS (
    
    SELECT  
        yyyymmdd,
        json_extract_scalar(event_props,'$.userId') AS customer_id,
        SUBSTR(profile_phone,3) customer_mobile,
        CAST(CAST(event_props_ct_session_id AS DECIMAL) AS VARCHAR) ct_session_id,
        updated_epoch screendropoff
    FROM 
        clevertap.customer_screendropoff_immutable
    WHERE
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND event_props_screen = 'PostOrderActivity'
    ),
    
    merge_datasets AS (
    
    SELECT 
        yyyymmdd,
        ride_distance,
        customer_id,
        order_id,
        order_status,
        requested,
        onTheWay,
        arrived,
        started,
        dropped,
        end_state,
        
        AVG(postorderstatus_requesting) postorderstatus_requesting,
        AVG(postorderstatus_ontheway) postorderstatus_ontheway,
        AVG(postorderstatus_arrived) postorderstatus_arrived,
        AVG(postorderstatus_started) postorderstatus_started,
        AVG(postorderstatus_dropped) postorderstatus_dropped,
        
        MAX(screendropoff_before_ontheway) screendropoff_before_ontheway,
        MAX(screendropoff_before_arrived) screendropoff_before_arrived,
        MAX(screendropoff_before_started) screendropoff_before_started,
        MAX(screendropoff_before_dropped) screendropoff_before_dropped,
        MAX(screendropoff_after_dropped) screendropoff_after_dropped,
        
        COUNT(DISTINCT screendropoff_before_ontheway) count_screendropoff_before_ontheway,
        COUNT(DISTINCT screendropoff_before_arrived) count_screendropoff_before_arrived,
        COUNT(DISTINCT screendropoff_before_started) count_screendropoff_before_started,
        COUNT(DISTINCT screendropoff_before_dropped) count_screendropoff_before_dropped,
        COUNT(DISTINCT screendropoff_after_dropped) count_screendropoff_after_dropped
        
    FROM 
        (
        SELECT 
            base.yyyymmdd,
            base.ride_distance,
            base.customer_id,
            base.customer_mobile,
            base.order_id,
            
            base.requested,
            COALESCE(pos.postorderstatus_requesting,base.requested) postorderstatus_requesting,
            CASE WHEN sdo.screendropoff >= base.requested AND (sdo.screendropoff < base.onTheWay OR sdo.screendropoff < pos.postorderstatus_ontheway) THEN sdo.screendropoff END screendropoff_before_ontheway,
            CASE WHEN base.incompleted_state >= base.requested AND base.incompleted_state < COALESCE(base.onTheWay, 0) THEN base.incompleted_state END end_before_onTheWay,
            
            base.onTheWay,
            pos.postorderstatus_ontheway,
            CASE WHEN (sdo.screendropoff >= base.onTheWay OR sdo.screendropoff >= pos.postorderstatus_ontheway) AND sdo.screendropoff < base.arrived THEN sdo.screendropoff END screendropoff_before_arrived,
            CASE WHEN base.incompleted_state >= base.onTheWay AND base.incompleted_state < COALESCE(base.arrived, 0) THEN base.incompleted_state END end_before_arrivded, 
            
            base.arrived,
            pos.postorderstatus_arrived,
            CASE WHEN (sdo.screendropoff >= base.arrived OR sdo.screendropoff >= pos.postorderstatus_arrived) AND sdo.screendropoff < base.started THEN sdo.screendropoff END screendropoff_before_started,
            CASE WHEN base.incompleted_state >= base.arrived AND base.incompleted_state < COALESCE(base.started, 0) THEN base.incompleted_state END end_after_arrivded, 
            
            base.started,
            pos.postorderstatus_started,
            CASE WHEN sdo.screendropoff >= base.started AND sdo.screendropoff < COALESCE(base.dropped, 0) THEN sdo.screendropoff END screendropoff_before_dropped,
            
            base.dropped,
            pos.postorderstatus_dropped,
            CASE WHEN sdo.screendropoff >= base.dropped AND sdo.screendropoff < base.end_state THEN sdo.screendropoff END screendropoff_after_dropped,
            
            base.end_state,
            base.order_status

        FROM 
            base_gross_order base
        LEFT JOIN 
            postorderstatus pos 
            ON base.yyyymmdd = pos.yyyymmdd
            AND base.order_id = pos.order_id
        LEFT JOIN 
            screendropoff sdo
            ON pos.yyyymmdd = sdo.yyyymmdd
            AND pos.customer_id = sdo.customer_id
            AND pos.customer_mobile = sdo.customer_mobile
            -- AND pos.ct_session_id = sdo.ct_session_id
        
        -- WHERE 
            -- base.order_id = '655b5956cbbabb6828039bbd'
        )
    GROUP BY 1,2,3,4,5,6,7,8,9,10,11
    ),
    
    calc_screen_time AS (
    
    SELECT
        *,
        CASE 
        WHEN requested_time = 0 THEN 0 
        WHEN time_requested_screen > requested_time THEN 100.0
        ELSE COALESCE(TRY(time_requested_screen*100.0/requested_time), 0) END AS requested_screen_percentage,

        CASE 
        WHEN onTheWay_time = 0 THEN 0 
        WHEN time_onTheWay_screen > onTheWay_time THEN 100.0
        ELSE COALESCE(TRY(time_onTheWay_screen*100.0/onTheWay_time), 0) END AS onTheWay_screen_percentage,
        
        CASE 
        WHEN arrived_time = 0 THEN 0 
        WHEN time_arrived_screen > arrived_time THEN 100.0
        ELSE COALESCE(TRY(time_arrived_screen*100.0/arrived_time), 0) END AS arrived_screen_percentage,
        
        CASE 
        WHEN started_time = 0 THEN 0 
        WHEN time_started_screen > started_time THEN 100.0
        ELSE COALESCE(TRY(time_started_screen*100.0/started_time), 0) END AS started_screen_percentage,
        
        CASE 
        WHEN ride_timing = 0 THEN 0
        WHEN (time_requested_screen + time_onTheWay_screen + time_arrived_screen + time_started_screen + time_dropped_screen) > ride_timing THEN 0
        ELSE COALESCE(TRY((time_requested_screen + time_onTheWay_screen + time_arrived_screen + time_started_screen + time_dropped_screen)*100.0/ride_timing),0) END AS screen_percentage
        
    FROM
        (
        SELECT
            merge_datasets.*,
            
            COALESCE(CASE 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NOT NULL AND screendropoff_before_ontheway <= onTheWay THEN abs(screendropoff_before_ontheway/1000 - requested/1000) 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL AND onTheWay IS NOT NULL AND postorderstatus_requesting <= onTheWay THEN abs(onTheWay/1000 - requested/1000) 
                WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL AND onTheWay IS NULL THEN abs(end_state/1000 - requested/1000) 
                END, 0) time_requested_screen,
                
            COALESCE((COALESCE(onTheWay,end_state)/1000 - requested/1000), 0) requested_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NOT NULL AND screendropoff_before_arrived <= arrived THEN abs(screendropoff_before_arrived/1000 - postorderstatus_ontheway/1000) 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL AND arrived IS NOT NULL AND postorderstatus_ontheway <= arrived THEN abs(arrived/1000 - postorderstatus_ontheway/1000) 
                WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL AND arrived IS NULL THEN abs(end_state/1000 - postorderstatus_ontheway/1000) 
                END, 0) time_onTheWay_screen,
                
            COALESCE((COALESCE(arrived,end_state)/1000 - onTheWay/1000), 0) onTheWay_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NOT NULL AND screendropoff_before_started <= started THEN abs(screendropoff_before_started/1000 - postorderstatus_arrived/1000) 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL AND started IS NOT NULL AND postorderstatus_arrived <= started THEN abs(started/1000 - postorderstatus_arrived/1000) 
                WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL AND started IS NULL THEN abs(end_state/1000 - postorderstatus_arrived/1000) 
                END, 0) time_arrived_screen,
                
            COALESCE((COALESCE(started,end_state)/1000 - arrived/1000), 0) arrived_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NOT NULL AND screendropoff_before_dropped <= dropped THEN abs(screendropoff_before_dropped/1000 - postorderstatus_started/1000) 
                WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NULL AND dropped IS NOT NULL AND postorderstatus_started <= dropped THEN abs(dropped/1000 - postorderstatus_started/1000)
                END, 0) time_started_screen,
                
            COALESCE((COALESCE(dropped,end_state)/1000 - started/1000), 0) started_time,
                
            COALESCE(CASE 
                WHEN postorderstatus_dropped IS NOT NULL AND screendropoff_after_dropped IS NOT NULL AND screendropoff_after_dropped > dropped THEN abs(screendropoff_after_dropped/1000 - postorderstatus_dropped/1000) 
                WHEN postorderstatus_dropped IS NOT NULL AND screendropoff_after_dropped IS NULL AND dropped IS NOT NULL AND postorderstatus_dropped > dropped THEN 15 
                END, 0) time_dropped_screen,
                
            ((GREATEST(COALESCE(screendropoff_after_dropped,0),dropped,end_state)/1000) - (requested/1000)) AS ride_timing,
            COALESCE((count_screendropoff_before_ontheway + count_screendropoff_before_arrived + count_screendropoff_before_started + count_screendropoff_before_dropped + count_screendropoff_after_dropped), 0) screendropoff
    
        FROM 
            merge_datasets
        )
    )
    
    SELECT
        
        ride_distance,
        
        -- requested
        COUNT(DISTINCT order_id) requested,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_requesting IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age viewed on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_requesting IS NOT NULL AND screendropoff_before_ontheway IS NULL THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age no dropOff on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_ontheway IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age 2-3 dropOff on requested screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_ontheway > 3 THEN order_id END)*100.0/COUNT(DISTINCT order_id)), 0) "%age 3+ dropOff on requested screen",
        
        (AVG(time_requested_screen))  "avg_time_on_requested_screen (sec)",
        (AVG(requested_screen_percentage)) "requested_screen_engagement %",
        
        -- onTheWay
        COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END) onTheWay,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_ontheway IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age viewed on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_ontheway IS NOT NULL AND screendropoff_before_arrived IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age no dropOff on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_arrived IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on onTheWay screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_arrived > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN onTheWay IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on onTheWay screen",
        
        -- AVG(count_screendropoff_before_arrived) count_screendropoff_before_arrived,
        (AVG(time_onTheWay_screen))  "avg_time_on_onTheWay_screen (sec)",
        (AVG(onTheWay_screen_percentage)) "onTheWay_screen_engagement %",
        
        -- arrived
        COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END) arrived,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_arrived IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age viewed on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_arrived IS NOT NULL AND screendropoff_before_started IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age no dropOff on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_started IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on arrived screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_started > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN arrived IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on arrived screen",
        
        (AVG(time_arrived_screen))  "avg_time_on_arrived_screen (sec)",
        (AVG(arrived_screen_percentage)) "arrived_screen_engagement %",
        
        -- started
        COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END) started,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_started IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age viewed on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_started IS NOT NULL AND screendropoff_before_dropped IS NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age no dropOff on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_dropped IN (2,3) THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age 2-3 dropOff on started screen",
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN count_screendropoff_before_dropped > 3 THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN started IS NOT NULL THEN order_id END)), 0) "%age 3+ dropOff on started screen",
        
        (AVG(time_started_screen))  "avg_time_on_started_screen (sec)",
        (AVG(started_screen_percentage)) "started_screen_engagement %",
        
        -- dropped
        COUNT(DISTINCT CASE WHEN dropped IS NOT NULL THEN order_id END) dropped,
        COALESCE(TRY(COUNT(DISTINCT CASE WHEN postorderstatus_dropped IS NOT NULL THEN order_id END)*100.0/COUNT(DISTINCT CASE WHEN dropped IS NOT NULL THEN order_id END)), 0) "%age viewed on dropped screen",
        (AVG(time_dropped_screen))  "avg_time_on_dropped_screen (sec)",
        
        --other
        AVG(screen_percentage) "overall_screen_engagement %",
        AVG(screendropoff) "avg screendropoff #"
        
    FROM 
        calc_screen_time
    WHERE 
        ride_distance != 'Other'
    GROUP BY 1
    ORDER BY 1

"""


df_ride_distance_view_data = pd.read_sql(ride_distance_view_data, connection)
df_ride_distance_view_data

,ride_distance,requested,%age viewed on requested screen,%age no dropOff on requested screen,%age 2-3 dropOff on requested screen,%age 3+ dropOff on requested screen,avg_time_on_requested_screen (sec),requested_screen_engagement %,onTheWay,%age viewed on onTheWay screen,%age no dropOff on onTheWay screen,%age 2-3 dropOff on onTheWay screen,%age 3+ dropOff on onTheWay screen,avg_time_on_onTheWay_screen (sec),onTheWay_screen_engagement %,arrived,%age viewed on arrived screen,%age no dropOff on arrived screen,%age 2-3 dropOff on arrived screen,%age 3+ dropOff on arrived screen,avg_time_on_arrived_screen (sec),arrived_screen_engagement %,started,%age viewed on started screen,%age no dropOff on started screen,%age 2-3 dropOff on started screen,%age 3+ dropOff on started screen,avg_time_on_started_screen (sec),started_screen_engagement %,dropped,%age viewed on dropped screen,avg_time_on_dropped_screen (sec),overall_screen_engagement %,avg screendropoff #
0,Long Distance,6829983,100.0,87.9,4.4,3.4,93.282026,89.7,5396197,77.9,27.9,21.1,15.4,166.290760,43.673160,4160511,54.6,31.1,9.7,2.1,31.376235,18.996586,3829476,55.8,6.2,21.2,11.2,161.357694,9.650207,3822604,34.0,4.481920,14.207044,2.701659
1,Short Distance,8569098,100.0,92.5,2.7,1.6,84.136318,92.2,6340625,70.8,26.9,19.1,10.8,113.329094,36.625726,5074002,47.9,27.3,8.4,1.4,19.181822,16.410769,4804935,44.8,6.5,16.4,4.4,51.528883,8.547336,4799510,33.6,4.288655,15.604052,1.806923


In [67]:
df_segment_view_data.round(2).to_clipboard(index=False)

In [68]:
df_ride_distance_view_data.round(2).to_clipboard(index=False)